# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-Ecp4o8BpOoN3pRxvq7gfXTFNvMU4X',
 'object': 'chat.completion',
 'created': 1792921510,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: A group of flamingos is called a flamboyance. Their pink color comes from carotenoid pigments in the food they eat. Want another fun fact on a different topic?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 559,
  'total_tokens': 570,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 512,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [4]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: A group of flamingos is called a flamboyance. Their pink color comes from carotenoid pigments in the food they eat. Want another fun fact on a different topic?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [5]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: A day on Venus is longer than a year on Venus. Venus spins very slowly (about 243 Earth days to rotate once) and orbits the Sun in about 225 Earth days, so a Venusian day is longer than its year. Want another fun fact in a different theme?'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [6]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [7]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [8]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2:1b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s a fun fact: There is a type of jellyfish that is immortal. The Turritopsis dohrnii, also known as the "immortal jellyfish," is a species of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back into its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal.'

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [16]:
# Step 1: Create your prompts

system_prompt = "You are a professional content Summarizer with a high sense of humor mixed with your snarky character, check what ever content sent to you and summarize it"
user_prompt = r"""
Summarize this ai skill reame file content: appsec-review
Threat-model-driven security reviews: full attack surface coverage, every finding verified by an independent pass and scored with CVSS 4.0.
Claude Code, Cursor and Codex load this one skill folder. Spring, Node.js, Python, PHP, .NET and Go back ends, browser front ends, mobile apps, Supabase and Firebase, CI pipelines, containers and infrastructure as code.

New to terms such as sink, coverage ledger, STRIDE or disproof? The plain-language glossary explains the 15 main concepts used by the workflow.

 Why it's different
Attack surface mapped by script.
A scripted inventory lists every entry point and dangerous sink. The ledger validator fails the run if any of them was skipped, so every run covers the same attack surface.	Threat-model driven.
Trust boundaries, data classes and a STRIDE pass, then abuse paths worked as an attacker would, either to the goal or to the line that stops them. Verified findings are chained into worse end-to-end exploits.
Findings are disproved, not just found.
A fresh verifier agent that never sees the finder's reasoning tries to refute each candidate. A rejection must cite the line of code that stops the attack, and rejected candidates are listed so you can overrule them.	It knows your framework.
Rules for Spring, Node, Python, PHP (Laravel, Symfony, WordPress), .NET, Go, web front ends, mobile apps, Supabase/Firebase, and CI, Docker, Kubernetes and Terraform, plus deep playbooks for BOLA/IDOR, payment callbacks, idempotency and race conditions.
Audit-ready output.
Each finding gives file:line, the exploit path, the impact and a fix using your own helpers, mapped to OWASP, ASVS, MASVS, CWE and PCI DSS. CVSS 4.0 vectors are computed by script, never estimated.	Built for the pipeline.
--fail-on <severity> gates merges, accepted risks expire on a date, and SARIF 2.1.0 export feeds GitHub code scanning or Jenkins.
 How it compares
appsec-review adapts its hunters, verifier and coverage ledger from Cloudflare's security-audit (NOTICE.md). It is also compared with Claude Code's built-in /security-review. All three make an agent try to disprove a finding before it is reported. They are built for different jobs:

Use appsec-review for PR reviews, web, API, mobile and Supabase/Firebase apps, audit deliverables, CI gates and a predictable cost. Use /security-review for a quick, zero-setup check of a branch's high-confidence issues. Use security-audit for native code, messaging-heavy systems, proving bugs by running the code, or the widest coverage of a large codebase when cost matters less.

Compared in October 2026 with security-audit at commit c1c8a8c and claude-code-security-review at commit 0c6a49f.

 Features
 Code review engine
Scripted inventory: lists every route, handler, resolver, consumer, screen, deep link, CI pipeline, internet-facing resource and dangerous sink per stack, so each run starts from the same ground (method/inventory.py)
Forward and backward assessment: follows each input from entry point to sink, while a sink tracer works back from every dangerous call (deep)
STRIDE and threat model: trust boundaries, data classes, attacker goals and abuse paths, each worked to its goal or to the line that stops it
Change-aware diff review: finds the commit that added a removed control, reads the callers of changed helpers, and flags security controls without a negative test
Library-aware: reads the source of the locked dependency version instead of guessing what it does
Chain pass: combines verified findings into worse end-to-end outcomes and verifies each hand-off
 Vulnerability coverage
Access control: IDOR/BOLA, function-level authorization, mass assignment, multi-tenancy, SSRF
Authentication and sessions: JWT, OAuth and OIDC, step-up and OTP, passwords, session handling, SAML, WebAuthn and passkeys, MFA downgrade, account recovery and linking, API keys, mTLS
Injection and input: SQL and LIKE, command, expression language, CRLF, XXE, deserialization, file handling
Business logic: workflow integrity, money movement, payment callbacks, idempotency, race conditions, replay, enumeration, resource limits
Web protocol: request smuggling, cache poisoning and deception, host and forwarded-header trust
Client side: DOM XSS, token storage, CSP, deep links, WebViews, mobile storage and platform APIs
Cryptography and data: secrets, crypto misuse, TLS, card and personal data (PCI DSS)
Configuration and supply chain: exposed tooling, headers and cookies, dependencies, CI, containers, IaC
AI and LLM agents: dangerous capability combinations, tool authorization, MCP identity, RAG and memory tenant isolation, model output handling, spend limits
Logging and errors: missing security events, secrets in logs, verbose errors, fail-open handling
 Independent verification
Hunters: wildcard, obvious-things and sink-trace passes look for what the main assessment missed
Fresh verifiers when available: a separate agent that never saw the finder's reasoning tries to disprove each candidate; Claude Code, Cursor and Codex use their native delegation
Sequential fallback: the same prompts run later in the coordinator session when subagents are disabled or unavailable, and the result labels them self-verified
Safe rejection rule: a critical or bug candidate cannot be recorded as rejected without a second independent agent; in a sequential run it stays needs_confirmation
Transparent rejections: rejected candidates are listed with the line that disproves them, so you can overrule them
 Reporting and CI
Audit-ready report: local HTML report with executive summary, threat model, remediation plan and appendix, plus a Markdown copy to feed to another session, both saved locally with their paths printed
CVSS 4.0 by script: every score computed by report/cvss40.py, never estimated
Coverage ledger: entry point - category units, validated against the inventory by method/ledger.py
CI gate and SARIF: --fail-on <severity>, time-limited accepted risks, SARIF export for code scanning
Multi-repository reviews: findings shared across repositories reported once as platform-wide
Benchmarks: evals/benchmark.py scores runs against known findings and decoys
 Supported stacks
Stack	Frameworks
Java and Kotlin	Spring Boot, MVC, WebFlux, Security, Data
Node.js and TypeScript	NestJS (REST and GraphQL), Express, Fastify, Koa, Hapi, Hono, and server code in Next.js and other full-stack frameworks
Python	Django, Django REST Framework, FastAPI, Flask, Starlette
PHP	Laravel (with Livewire), Symfony, WordPress, Slim, CodeIgniter, CakePHP, Laminas
.NET	ASP.NET Core MVC, Web API, minimal APIs, Blazor
Go	net/http, Gin, Echo, Fiber, chi, gRPC
Browser front ends	React, Next.js, Angular, Vue, Nuxt, Svelte, Solid, Astro
Mobile	React Native, Expo, Flutter, native Android and iOS
Backend as a service	Supabase, Firebase
CI, containers and IaC	GitHub Actions, GitLab CI, Jenkins, Docker, Kubernetes, Helm, Terraform
Each stack has a rule file in rules/stacks/ and inventory patterns that list its entry points and sinks. Code in other stacks gets only the general category checks, without a scripted inventory, so coverage is weaker; the report says when no stack file matched.

 Standards covered
Important

Every finding is mapped to the standards below, and the report lists the ones each review was based on.

Standard	Used for
OWASP Top 10:2025	Web application risk categories
OWASP API Security Top 10:2023	API risks: BOLA, broken function-level authorization, mass assignment
OWASP ASVS 5.0 (incl. V3 Web Frontend Security)	Verification requirements for back ends and browser front ends
OWASP WSTG-CLNT	Client-side testing for web front ends
OWASP MASVS, MASTG, MASWE	Mobile app security requirements and testing
OWASP Mobile Top 10:2024	Mobile risk categories
CWE Top 25	Root-cause weakness IDs on every finding
PCI DSS	Card data handling and payment flows
NIST SP 800-63B-4	Passwords, authenticators and session rules
RFC 9700	OAuth 2.0 security best current practice
RFC 8252	OAuth 2.0 for native apps
RFC 10017	OAuth 2.0 for browser-based apps
CVSS 4.0	Severity scoring, computed by script
 Quick start
Install the skill once, then invoke it inside the repository you want reviewed. Exact commands, sequential mode and the test boundary are in docs/platforms.md.

git clone https://github.com/ibrahimabdallatif/appsec-review.git
cd appsec-review
python -I skills/appsec-review/method/install_links.py
That links skills/appsec-review/ into the Claude Code, Cursor and Codex user skill directories. It does not copy the methodology. Existing links are left alone unless --replace is passed; a real directory is never deleted. The link installer is tested on Linux and Windows, using Windows junctions when symlink permission is unavailable.

Or copy via the skills CLI (this writes a snapshot, not a link):

npx skills add ibrahimabdallatif/appsec-review --skill appsec-review -g -a claude-code -a cursor -a codex -y
Host	Invoke
Claude Code	/appsec-review
Cursor	/appsec-review
Codex	/skills, then appsec-review, or $appsec-review
/appsec-review                               # current branch vs. its base, plus uncommitted changes
/appsec-review whole repo                    # the whole repository
/appsec-review deep                          # in-depth review for high-stakes code
Add "sequential" to skip subagents. The same steps still run.

Tip

Start with a plain /appsec-review on a branch. It is the fastest mode and shows you the output format before you spend time on a whole repository. More commands: Getting started.

The skill runs only when you invoke it (disable-model-invocation: true on Claude Code and Cursor). Codex does not document that flag; the description tells it not to run for a general security question.

 Compatibility
The shared skill, discovery adapters, sequential fallback and installation scripts are implemented. The automated suite currently passes on Ubuntu and Windows with Python 3.10 and 3.13: 471 unit tests plus 13 FIRST CVSS 4.0 reference vectors. Live skill invocation was not re-run on every host, so discovery files are described as implemented rather than end-to-end tested. Details and exact boundaries are in docs/platforms.md.

Requirement	Details
Claude Code	Skills and subagents. Project adapter .claude/skills/appsec-review/. User install ~/.claude/skills/appsec-review. Headless CI example: claude -p in docs/ci-pipeline.md
Cursor	Project adapters .cursor/skills/appsec-review/ and .agents/skills/appsec-review/. User install ~/.cursor/skills/appsec-review. Subagents when the session can spawn them; otherwise the sequential fallback
Codex	Project adapter .agents/skills/appsec-review/. User install ~/.agents/skills/appsec-review and ~/.codex/skills/appsec-review. Subagents when the session can spawn them; otherwise the sequential fallback
Python	3.10+ on your PATH as python, for the helper scripts (standard library only, tested on 3.10 and 3.13)
Git	Needed for diff, branch and PR reviews and for multi-repository discovery
Node.js	Only to install with npx skills add. Not required for install_links.py or for a review
OS	Linux, macOS and Windows (tests run on Linux and Windows)
Optional tools	gitleaks, trivy or osv-scanner, semgrep, your ecosystem's audit tool and, for pipelines and infrastructure, zizmor, actionlint, checkov and hadolint; used when installed, never installed by the skill
Report publishing	Local HTML and Markdown always, under ~/.appsec-review/<repo>/. A private claude.ai artifact only when you explicitly ask, and only on Claude Code. Cursor and Codex do not publish
 Modes
Mode	Use it for	Subagents	Coverage ledger
diff	A branch, PR or files (default)	None, except one verifier for a surviving critical	None
standard	A whole repository, an endpoint, or several repositories	One hunter, one batch verifier, one reviewer for rejected serious candidates	Module - category
deep	High-stakes code, only when you say "deep"	Two hunters (incl. a backward sink tracer), one verifier per serious candidate	Entry point - category
Every mode reads all in-scope code and follows each input to its sinks; modes change how many agents run and how detailed the ledger is. Say "sequential", or set APPSEC_REVIEW_PARALLEL=0, to run those agent steps in the same session. deep takes several times longer and uses much more tokens than standard.

 What you get
Findings ordered by severity (critical, bug, warning, suggestion), each one like this (illustrative):

[Security] Order id not bound to caller (API1, CWE-639)        critical
src/orders/OrderController.java:42

Attack: any signed-in user sends GET /orders/id with another customer's order id.
Impact: read access to every customer's orders.
Fix:    load through orderRepository.findByIdAndCustomerId(id, principal.customerId()).
Then the "needs confirmation" findings with the exact fact to check, what was checked and found clean, the coverage summary, the candidates rejected by verification, and a run summary with time and tokens.

standard and deep runs also write a report (a diff run asks first) with CVSS 4.0 scores and standards references for each finding to ~/.appsec-review/<repo>/: report.html for people and report.md, the same content in Markdown, to feed to another session ("read report.md and fix finding 4.1.2"). The run prints the paths from method/review_paths.py, which joins the home directory and .appsec-review with a real separator. A private claude.ai artifact is added only when you ask to publish, and only from Claude Code:

Report
- HTML: C:\Users\<you>\.appsec-review\cards-service\report.html
- Markdown: C:\Users\<you>\.appsec-review\cards-service\report.md
CI and other headless runs write the same local files and do not publish.

 In CI
The review always runs to completion. A separate gate reads the finished ledger and fails the build on findings at or above a severity:

python -I <skill folder>/method/ledger.py ~/.appsec-review/<repo>/coverage-ledger.json \
  --inventory ~/.appsec-review/<repo>/inventory.json --fail-on critical
Exit codes: 0 pass, 3 blocking findings, 1 incomplete review or invalid ledger, 2 bad arguments. Add --accepted <file> to waive findings the team has accepted, each until an expiry date and up to a severity; waived findings are still printed. --sarif <file> exports the findings for GitHub code scanning or Jenkins. Agent setup, accepted risks, SARIF, a Jenkins example and scheduling advice: docs/ci-pipeline.md.

Warning

Ledgers, inventories and reports describe how to attack open issues. They are kept in ~/.appsec-review/, outside your repository. Nothing is published unless you ask. Check who can see a report before sharing.

"""

# Step 2: Make the messages list

messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ] 

# Step 3: Call Ollama chat.completions

# OLLAMA_BASE_URL = "http://localhost:11434/v1"
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
response = ollama.chat.completions.create(model="llama3.2:1b", messages = messages)


# Step 4: print the result
print(response.choices[0].message.content)

**Summary of appsec-review**

Appsec-review is a threat-model-driven security review tool that provides a comprehensive and detailed security review of applications, websites, and APIs. It is used to identify vulnerabilities, and then to provide a report with recommendations for mitigation.

**Key Features**

* Provides full attack surface coverage, with every finding verified by an independent pass and scored with CVSS 4.0.
* Supports a wide range of stack frameworks, including Spring, Node.js, Python, PHP, .NET, Go, and more.
* Uses a scripted inventory to list every entry point and danger sink, so that the review runs from the same ground as the application.
* Includes a STRIDE and threat model pass, which works to its goal or to the line that stops it.
* Supports hunters (wildcard, obvious-things, and sink-trace passes) for quick assessments and a fresh verifier agent that never sees the finder's reasoning.
* Includes a sequence fall back, which allows for the same prompts to run l